# Experiment 12: Big Data Processing Pipeline (Power BI)
**Aim:** Implement an end-to-end Big Data pipeline using Apache Spark to ingest, process, and analyze a real dataset, and visualize the results using Power BI.

### Step 1: Start Spark Session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataMiniProject") \
    .getOrCreate()
spark

### Step 2: Data Ingestion (Load NYC Taxi Dataset)

In [ ]:
df = spark.read.csv(
    "nyc_taxi.csv",
    header=True,
    inferSchema=True
)

df.show(5)

### Step 3: Data Cleaning (Null and Invalid Trips)

In [ ]:
clean_df = df.dropna()

clean_df = clean_df.filter(
    (clean_df.trip_distance > 0) &
    (clean_df.fare_amount > 0)
)
print("Records cleaned successfully")

### Step 4: Attribute Extraction

In [ ]:
from pyspark.sql.functions import col

feature_df = clean_df.select(
    col("tpep_pickup_datetime").alias("pickup_datetime"),
    col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    col("passenger_count"),
    col("trip_distance"),
    col("fare_amount"),
    col("payment_type")
)
feature_df.printSchema()

### Step 5: Feature Engineering & Step 6: Data Analytics

In [ ]:
from pyspark.sql.functions import to_date, avg

# Average Fare Amount Per Day
daily_avg_fare = feature_df.withColumn(
    "pickup_date",
    to_date("pickup_datetime")
).groupBy("pickup_date") \
.agg(avg("fare_amount").alias("avg_fare"))

daily_avg_fare.show(5)

### Step 7: Store Processed Data (Export for Power BI)
We convert the grouped Spark DataFrame to a Pandas DataFrame to dump it natively as a single static CSV for Power BI.

In [ ]:
import os

# Create temp output directory natively if missing
if not os.path.exists("C:/temp"):
    os.makedirs("C:/temp", exist_ok=True)

pdf = daily_avg_fare.toPandas()
pdf.to_csv("C:/temp/daily_avg_fare.csv", index=False)
print("Data successfully written to output folder (C:/temp/daily_avg_fare.csv)")